# Freelancer Bio Quality Classifier

This notebook implements a machine learning model to classify the quality of freelancer bios. The model will determine if a bio is properly written or not, and provide feedback for improvement.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pickle

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\Siddhesh
[nltk_data]     Patil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Siddhesh
[nltk_data]     Patil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 1. Data Loading and Exploration

In [2]:
# Load the freelancer dataset
df = pd.read_csv('freelancers_dataset_full.csv')

# Display the first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 7)


,Freelancer_ID,Skills,Experience_Years,Description,Project_Rate,Rating,Jobs_Completed
0,F0001,"color palette, adobe lightroom, print design",11,Senior Graphic & Visual Design professional wi...,$3675-$8602,4.9,150
1,F0002,"sql, analytics",4,Mid-level Data Science professional with 4 yea...,$690-$1771,4.6,45
2,F0003,"data mining, statistics, data processing",2,Entry-level Data Science professional with 2 y...,$59-$605,3.5,8
3,F0004,"sql programming, database architecture, sql",14,Senior Database Management professional with 1...,$3570-$5749,4.6,132
4,F0005,"responsive design, user interface design",10,Senior UI/UX Design professional with 10 years...,$1650-$9117,4.6,94


In [3]:
# Check for missing values
print("Missing values:")
df.isnull().sum()

Missing values:


Freelancer_ID       0
Skills              0
Experience_Years    0
Description         0
Project_Rate        0
Rating              0
Jobs_Completed      0
dtype: int64

In [4]:
# Basic statistics of the dataset
df.describe(include='all')

,Freelancer_ID,Skills,Experience_Years,Description,Project_Rate,Rating,Jobs_Completed
count,10000,10000,10000.000000,10000,10000,10000.000000,10000.000000
unique,10000,9670,NaN,9975,9988,NaN,NaN
top,F0001,"user experience design, prototyping",NaN,Senior Mobile Development professional with 12...,$1189-$2375,NaN,NaN
freq,1,6,NaN,2,2,NaN,NaN
mean,NaN,NaN,7.977100,NaN,NaN,4.506380,75.790300
std,NaN,NaN,4.335834,NaN,NaN,0.358896,62.358906
min,NaN,NaN,1.000000,NaN,NaN,3.500000,1.000000
25%,NaN,NaN,4.000000,NaN,NaN,4.300000,17.000000
50%,NaN,NaN,8.000000,NaN,NaN,4.600000,59.000000
75%,NaN,NaN,12.000000,NaN,NaN,4.800000,130.000000


## 2. Data Preprocessing and Feature Engineering

In [5]:
# Clean and preprocess the bio descriptions
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply text cleaning
df['clean_description'] = df['Description'].apply(clean_text)

In [6]:
# Feature engineering: Create quality labels based on bio characteristics
def label_bio_quality(text, min_length=50):
    if pd.isna(text) or not isinstance(text, str):
        return 0  # Poor quality if empty or not a string
    
    # Check length
    if len(text) < min_length:
        return 0  # Too short
    
    # Check word count
    words = text.split()
    if len(words) < 10:
        return 0  # Too few words
    
    # Check for keyword diversity (skills, experience, etc.)
    keywords = ['experience', 'skill', 'project', 'develop', 'design', 'create', 'work', 'professional']
    keyword_count = sum(1 for keyword in keywords if keyword in text.lower())
    
    if keyword_count < 2:
        return 0  # Poor keyword diversity
    
    return 1  # Good quality

# Apply quality labeling
df['bio_quality'] = df['Description'].apply(label_bio_quality)

# Display distribution of quality labels
print("Bio Quality Distribution:")
print(df['bio_quality'].value_counts())
print(f"Percentage of good quality bios: {df['bio_quality'].mean() * 100:.2f}%")

Bio Quality Distribution:
bio_quality
1    10000
Name: count, dtype: int64
Percentage of good quality bios: 100.00%


## 3. Feature Extraction

In [7]:
# Split the data into training and testing sets
X = df['clean_description']
y = df['bio_quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 8000
Testing set size: 2000


In [8]:
# Create a TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, 
                                  stop_words='english', 
                                  ngram_range=(1, 2))

# Transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF feature matrix shape: {X_train_tfidf.shape}")

TF-IDF feature matrix shape: (8000, 5000)


## 4. Model Training and Evaluation

In [9]:
# Train a Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluate the model
print("Logistic Regression Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_lr):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 1

In [ ]:
# Train a Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test_tfidf)

# Evaluate the model
print("Random Forest Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 5. Model Selection and Saving

In [ ]:
# Compare models and select the best one
lr_accuracy = accuracy_score(y_test, y_pred_lr)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
lr_f1 = f1_score(y_test, y_pred_lr)
rf_f1 = f1_score(y_test, y_pred_rf)

print("Model Comparison:")
print(f"Logistic Regression - Accuracy: {lr_accuracy:.4f}, F1 Score: {lr_f1:.4f}")
print(f"Random Forest - Accuracy: {rf_accuracy:.4f}, F1 Score: {rf_f1:.4f}")

# Select the best model based on F1 score
if rf_f1 > lr_f1:
    best_model = rf_model
    print("\nRandom Forest selected as the best model.")
else:
    best_model = lr_model
    print("\nLogistic Regression selected as the best model.")

In [ ]:
# Save the model and vectorizer
model_filename = 'bio_quality_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

with open(model_filename, 'wb') as file:
    pickle.dump(best_model, file)
    
with open(vectorizer_filename, 'wb') as file:
    pickle.dump(tfidf_vectorizer, file)
    
print(f"Model saved to {model_filename}")
print(f"Vectorizer saved to {vectorizer_filename}")

## 6. Bio Quality Analysis Function

In [ ]:
def analyze_bio_quality(bio_text):
    """
    Analyze the quality of a freelancer bio and provide feedback.
    
    Args:
        bio_text (str): The bio text to analyze
        
    Returns:
        dict: Analysis results including quality score and feedback
    """
    # Clean the text
    clean_bio = clean_text(bio_text)
    
    # Transform using the vectorizer
    bio_features = tfidf_vectorizer.transform([clean_bio])
    
    # Predict quality
    quality_score = best_model.predict(bio_features)[0]
    quality_prob = best_model.predict_proba(bio_features)[0][1]  # Probability of good quality
    
    # Generate feedback
    feedback = []
    
    # Check length
    if len(bio_text) < 50:
        feedback.append("Your bio is too short. Aim for at least 100-150 characters.")
    
    # Check word count
    words = bio_text.split()
    if len(words) < 10:
        feedback.append("Your bio has too few words. Include at least 20-30 words.")
    
    # Check for keyword inclusion
    keywords = {
        'experience': "Mention your years of experience in your field.",
        'skill': "Include your key skills relevant to your profession.",
        'project': "Briefly mention significant projects you've worked on.",
        'professional': "Highlight your professional background."
    }
    
    missing_keywords = []
    for keyword, suggestion in keywords.items():
        if keyword not in bio_text.lower():
            missing_keywords.append(suggestion)
    
    if missing_keywords:
        feedback.extend(missing_keywords)
    
    # Overall assessment
    if quality_score == 1:
        overall = "Your bio appears to be well-written."
        if feedback:
            overall += " However, there are still some improvements you could make."
    else:
        overall = "Your bio needs improvement to better showcase your skills and experience."
    
    return {
        "quality_score": int(quality_score),
        "quality_probability": float(quality_prob),
        "overall_assessment": overall,
        "feedback": feedback
    }

In [ ]:
# Test the analysis function with sample bios
sample_bios = [
    "I am a developer.",  # Poor bio
    "Experienced web developer with 5 years of experience in React, Node.js, and Python. Completed over 20 projects for clients in e-commerce and finance sectors. Professional approach to problem-solving and client communication."  # Good bio
]

for i, bio in enumerate(sample_bios):
    print(f"\nSample Bio {i+1}:\n{bio}")
    analysis = analyze_bio_quality(bio)
    print(f"\nQuality Score: {analysis['quality_score']}")
    print(f"Quality Probability: {analysis['quality_probability']:.2f}")
    print(f"Overall Assessment: {analysis['overall_assessment']}")
    if analysis['feedback']:
        print("Feedback:")
        for item in analysis['feedback']:
            print(f"- {item}")
    else:
        print("No specific feedback - your bio looks good!")

## 7. Integration with Flask Application

In [ ]:
# Example code for integrating with the Flask application
"""
# In your Flask application (app.py):

import pickle
from bio_analyzer import analyze_bio_quality, clean_text

# Load the model and vectorizer
with open('bio_quality_model.pkl', 'rb') as file:
    bio_model = pickle.load(file)
    
with open('tfidf_vectorizer.pkl', 'rb') as file:
    tfidf_vectorizer = pickle.load(file)

# Add a route to check bio quality
@app.route('/check_bio_quality', methods=['POST'])
def check_bio_quality():
    data = request.get_json()
    bio_text = data.get('bio', '')
    
    analysis = analyze_bio_quality(bio_text)
    
    return jsonify(analysis)
"""